# Sample Data Generator for Nucleus Segmentation

Gera amostras reduzidas dos datasets processados (MoNuSeg, PanNuke, NuInSeg).

## Fluxo de Trabalho
1. **Instalar dependências** (primeira execução)
2. **Configurar parâmetros** de geração
3. **Gerar amostras** dos datasets

**Pré-requisito**: Executar `00_Installation.ipynb` para processar os dados brutos

## 1. Setup e Instalação

In [28]:
import subprocess
import sys

packages = ['pandas>=2.0.0', 'numpy>=1.26.0']

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✓ Todas as dependências instaladas com sucesso!')

✓ Todas as dependências instaladas com sucesso!


In [29]:
import pandas as pd
import numpy as np
import os
import pathlib
import shutil
import logging
from typing import Dict, List, Tuple
from pathlib import Path

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Diretório de trabalho
WORKING_DIRECTORY = Path(os.getcwd())
print(f'Diretório de trabalho: {WORKING_DIRECTORY}')
print(f'✓ Importações concluídas')

Diretório de trabalho: c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\src
✓ Importações concluídas


## 2. Definição das Classes

In [30]:
class SampleGenerator:
    """Gera amostras reduzidas dos datasets processados."""

    def __init__(self, working_dir: Path = None, output_dir: Path = None):
        """
        Inicializar o gerador de amostras.
        
        Args:
            working_dir: Diretório de trabalho (padrão: diretório atual)
            output_dir: Diretório de saída (padrão: data/samples)
        """
        self.working_dir = working_dir or Path(os.getcwd())
        self.interim_dir = self.working_dir.parent / 'data' / 'interim'
        self.output_dir = output_dir or (self.working_dir.parent / 'data' / 'samples' / 'raw')
        
        logger.info(f'Interim dir: {self.interim_dir}')
        logger.info(f'Output dir: {self.output_dir}')
        
    def create_sample_dir(self, dataset_name: str) -> Path:
        """Criar estrutura de diretórios para um dataset."""
        sample_path = self.output_dir / dataset_name
        (sample_path / 'Images').mkdir(parents=True, exist_ok=True)
        (sample_path / 'Masks').mkdir(parents=True, exist_ok=True)
        logger.info(f"Diretório criado: {sample_path}")
        return sample_path
        
    def generate_monuseg_samples(self, num_samples: int = 5) -> bool:
        """Gerar amostras do MoNuSeg."""
        logger.info(f"\nGerando amostras MoNuSeg (n={num_samples})...")
        
        source_dir = self.interim_dir / 'MoNuSeg'
        if not source_dir.exists():
            logger.warning(f"Diretório não encontrado: {source_dir}")
            return False
            
        try:
            labels_file = source_dir / 'labels.csv'
            if not labels_file.exists():
                logger.warning(f"Arquivo de rótulos não encontrado: {labels_file}")
                return False
                
            labels_df = pd.read_csv(labels_file)
            
            # Selecionar amostras aleatórias
            if len(labels_df) < num_samples:
                logger.warning(f"Solicitados {num_samples} mas apenas {len(labels_df)} disponíveis")
                selected_samples = labels_df['FileName'].tolist()
            else:
                selected_samples = labels_df['FileName'].sample(n=num_samples, random_state=42).tolist()
            
            sample_path = self.create_sample_dir('MoNuSeg')
            
            source_images = source_dir / 'Images'
            source_masks = source_dir / 'Masks'
            
            for filename in selected_samples:
                img_file = source_images / f'{filename}.tif'
                mask_file = source_masks / f'{filename}.xml'
                
                if img_file.exists() and mask_file.exists():
                    shutil.copy(img_file, sample_path / 'Images' / f'{filename}.tif')
                    shutil.copy(mask_file, sample_path / 'Masks' / f'{filename}.xml')
            
            sample_labels = labels_df[labels_df['FileName'].isin(selected_samples)]
            sample_labels.to_csv(sample_path / 'labels.csv', index=False)
            
            logger.info(f"✓ MoNuSeg: {len(selected_samples)} amostras salvas em {sample_path}")
            return True
            
        except Exception as e:
            logger.error(f"Erro ao gerar amostras MoNuSeg: {e}")
            return False
    
    def generate_pannuke_samples(self, num_samples: int = 10) -> bool:
        """Gerar amostras do PanNuke."""
        logger.info(f"\nGerando amostras PanNuke (n={num_samples} por fold)...")
        
        source_dir = self.interim_dir / 'PanNuke'
        if not source_dir.exists():
            logger.warning(f"Diretório não encontrado: {source_dir}")
            return False
            
        try:
            labels_file = source_dir / 'labels.csv'
            if not labels_file.exists():
                logger.warning(f"Arquivo de rótulos não encontrado: {labels_file}")
                return False
                
            sample_path = self.create_sample_dir('PanNuke')
            
            images_dir = source_dir / 'Images'
            masks_dir = source_dir / 'Masks'
            
            selected_indices = []
            
            for fold_idx in range(1, 4):
                img_file = images_dir / f'fold{fold_idx}.npy'
                mask_file = masks_dir / f'fold{fold_idx}.npy'
                
                if not (img_file.exists() and mask_file.exists()):
                    continue
                
                try:
                    images = np.load(img_file)
                    masks = np.load(mask_file)
                    
                    fold_size = images.shape[0]
                    sample_per_fold = min(num_samples, fold_size)
                    indices = np.random.RandomState(42).choice(
                        fold_size, size=sample_per_fold, replace=False
                    )
                    
                    sample_images = images[indices]
                    sample_masks = masks[indices]
                    
                    np.save(
                        sample_path / 'Images' / f'fold{fold_idx}_sample.npy',
                        sample_images
                    )
                    np.save(
                        sample_path / 'Masks' / f'fold{fold_idx}_sample.npy',
                        sample_masks
                    )
                    
                    logger.info(f"  Fold {fold_idx}: {len(indices)} amostras extraídas (shape: {sample_images.shape})")
                    selected_indices.extend([fold_idx] * len(indices))
                    
                except Exception as e:
                    logger.warning(f"Erro ao processar fold {fold_idx}: {e}")
                    continue
            
            metadata = {
                'fold': selected_indices,
                'sample_index': list(range(len(selected_indices)))
            }
            metadata_df = pd.DataFrame(metadata)
            metadata_df.to_csv(sample_path / 'labels.csv', index=False)
            
            logger.info(f"✓ PanNuke: {len(selected_indices)} amostras salvas em {sample_path}")
            return True
            
        except Exception as e:
            logger.error(f"Erro ao gerar amostras PanNuke: {e}")
            return False
    
    def generate_nuinseg_samples(self, num_samples: int = 5) -> bool:
        """Gerar amostras do NuInSeg."""
        logger.info(f"\nGerando amostras NuInSeg (n={num_samples})...")
        
        source_dir = self.interim_dir / 'NuInSeg'
        if not source_dir.exists():
            logger.warning(f"Diretório não encontrado: {source_dir}")
            return False
            
        try:
            labels_file = source_dir / 'labels.csv'
            if not labels_file.exists():
                logger.warning(f"Arquivo de rótulos não encontrado: {labels_file}")
                return False
                
            labels_df = pd.read_csv(labels_file)
            
            if len(labels_df) < num_samples:
                logger.warning(f"Solicitados {num_samples} mas apenas {len(labels_df)} disponíveis")
                selected_indices = list(range(len(labels_df)))
            else:
                selected_indices = np.random.RandomState(42).choice(
                    len(labels_df), size=num_samples, replace=False
                ).tolist()
            
            sample_path = self.create_sample_dir('NuInSeg')
            
            source_images = source_dir / 'Images'
            source_masks = source_dir / 'Masks'
            
            selected_rows = []
            
            for idx in selected_indices:
                row = labels_df.iloc[idx]
                filename = row['FileName']
                
                img_file = source_images / f'{filename}.png'
                mask_file = source_masks / f'{filename}.png'
                
                if img_file.exists() and mask_file.exists():
                    shutil.copy(img_file, sample_path / 'Images' / f'{filename}.png')
                    shutil.copy(mask_file, sample_path / 'Masks' / f'{filename}.png')
                    selected_rows.append(row)
            
            if selected_rows:
                sample_labels = pd.DataFrame(selected_rows)
                sample_labels = sample_labels.reset_index(drop=True)
                sample_labels.to_csv(sample_path / 'labels.csv', index=False)
            
            logger.info(f"✓ NuInSeg: {len(selected_rows)} amostras salvas em {sample_path}")
            return True
            
        except Exception as e:
            logger.error(f"Erro ao gerar amostras NuInSeg: {e}")
            return False
    
    def generate_all_raw_samples(self, monuseg: int = 5, pannuke: int = 10, nuinseg: int = 5) -> Dict[str, bool]:
        """Gerar amostras para todos os datasets."""
        logger.info("\n" + "="*60)
        logger.info("Iniciando Geração de Amostras")
        logger.info("="*60)
        
        results = {
            'MoNuSeg': self.generate_monuseg_samples(monuseg),
            'PanNuke': self.generate_pannuke_samples(pannuke),
            'NuInSeg': self.generate_nuinseg_samples(nuinseg),
        }
        
        logger.info("\n" + "="*60)
        logger.info("Resumo da Geração de Amostras")
        logger.info("="*60)
        for dataset, success in results.items():
            status = "✓ SUCESSO" if success else "✗ FALHA"
            logger.info(f"{dataset}: {status}")
        
        logger.info(f"\nAmostras salvas em: {self.output_dir}")
        
        return results

print('✓ Classe SampleGenerator definida com sucesso')

✓ Classe SampleGenerator definida com sucesso


In [31]:
class DatasetInfo:
    """Exibir informações sobre os datasets."""
    
    @staticmethod
    def show_monuseg_info(data_dir: Path) -> None:
        """Exibir informações do MoNuSeg."""
        dataset_path = data_dir / 'interim' / 'MoNuSeg'
        
        if not dataset_path.exists():
            print(f"⚠ Dados MoNuSeg não encontrados: {dataset_path}")
            return
        
        print("\n" + "="*50)
        print("Informações do MoNuSeg")
        print("="*50)
        
        images_dir = dataset_path / 'Images'
        masks_dir = dataset_path / 'Masks'
        labels_file = dataset_path / 'labels.csv'
        
        if labels_file.exists():
            labels_df = pd.read_csv(labels_file)
            print(f"Total de imagens: {len(labels_df)}")
            
            image_count = len(list(images_dir.glob('*.tif')))
            mask_count = len(list(masks_dir.glob('*.xml')))
            
            print(f"Arquivos de imagem (.tif): {image_count}")
            print(f"Arquivos de máscara (.xml): {mask_count}")
    
    @staticmethod
    def show_pannuke_info(data_dir: Path) -> None:
        """Exibir informações do PanNuke."""
        dataset_path = data_dir / 'interim' / 'PanNuke'
        
        if not dataset_path.exists():
            print(f"⚠ Dados PanNuke não encontrados: {dataset_path}")
            return
        
        print("\n" + "="*50)
        print("Informações do PanNuke")
        print("="*50)
        
        images_dir = dataset_path / 'Images'
        masks_dir = dataset_path / 'Masks'
        labels_file = dataset_path / 'labels.csv'
        
        total_images = 0
        
        for fold in range(1, 4):
            img_file = images_dir / f'fold{fold}.npy'
            mask_file = masks_dir / f'fold{fold}.npy'
            
            if img_file.exists() and mask_file.exists():
                try:
                    images = np.load(img_file)
                    masks = np.load(mask_file)
                    
                    print(f"Fold {fold}:")
                    print(f"  Shape das imagens: {images.shape}")
                    print(f"  Shape das máscaras: {masks.shape}")
                    
                    total_images += images.shape[0]
                except Exception as e:
                    print(f"  Erro ao ler fold {fold}: {e}")
        
        if labels_file.exists():
            labels_df = pd.read_csv(labels_file)
            print(f"\nAmostras com rótulos: {len(labels_df)}")
        
        print(f"Total de imagens nos folds: {total_images}")
    
    @staticmethod
    def show_nuinseg_info(data_dir: Path) -> None:
        """Exibir informações do NuInSeg."""
        dataset_path = data_dir / 'interim' / 'NuInSeg'
        
        if not dataset_path.exists():
            print(f"⚠ Dados NuInSeg não encontrados: {dataset_path}")
            return
        
        print("\n" + "="*50)
        print("Informações do NuInSeg")
        print("="*50)
        
        images_dir = dataset_path / 'Images'
        masks_dir = dataset_path / 'Masks'
        labels_file = dataset_path / 'labels.csv'
        
        image_count = len(list(images_dir.glob('*.png')))
        mask_count = len(list(masks_dir.glob('*.png')))
        
        print(f"Arquivos de imagem (.png): {image_count}")
        print(f"Arquivos de máscara (.png): {mask_count}")
        
        if labels_file.exists():
            labels_df = pd.read_csv(labels_file)
            print(f"Amostras com rótulos: {len(labels_df)}")
            
            if 'Tissue' in labels_df.columns:
                print("\nDistribuição de tipos de tecido:")
                tissue_counts = labels_df['Tissue'].value_counts()
                for tissue, count in tissue_counts.items():
                    print(f"  {tissue}: {count}")

    @staticmethod
    def show_processed_monuseg_info(data_dir: Path) -> None:
        """Exibir informações do MoNuSeg processado (patches)."""
        dataset_path = data_dir / 'processed' / 'MoNuSeg'
        
        if not dataset_path.exists():
            print(f"⚠ Dados processados MoNuSeg não encontrados: {dataset_path}")
            return
        
        print("\n" + "="*50)
        print("Informações do MoNuSeg Processado")
        print("="*50)
        
        images_dir = dataset_path / 'Images'
        masks_dir = dataset_path / 'Masks'
        labels_file = dataset_path / 'labels.csv'
        
        image_count = len(list(images_dir.glob('*.npy'))) if images_dir.exists() else 0
        mask_count = len(list(masks_dir.glob('*.npy'))) if masks_dir.exists() else 0
        
        print(f"Patches de imagem (.npy): {image_count}")
        print(f"Patches de máscara (.npy): {mask_count}")
        
        if labels_file.exists():
            labels_df = pd.read_csv(labels_file)
            print(f"Amostras com rótulos: {len(labels_df)}")

    @staticmethod
    def show_processed_pannuke_info(data_dir: Path) -> None:
        """Exibir informações do PanNuke processado (arrays por fold)."""
        dataset_path = data_dir / 'processed' / 'PanNuke'
        
        if not dataset_path.exists():
            print(f"⚠ Dados processados PanNuke não encontrados: {dataset_path}")
            return
        
        print("\n" + "="*50)
        print("Informações do PanNuke Processado")
        print("="*50)
        
        images_dir = dataset_path / 'Images'
        masks_dir = dataset_path / 'Masks'
        
        total_patches = 0
        
        for fold in range(1, 4):
            img_file = images_dir / f'fold{fold}.npy'
            mask_file = masks_dir / f'fold{fold}.npy'
            
            if img_file.exists() and mask_file.exists():
                try:
                    images = np.load(img_file, mmap_mode='r')
                    masks = np.load(mask_file, mmap_mode='r')
                    
                    print(f"Fold {fold}:")
                    print(f"  Shape das imagens: {images.shape}")
                    print(f"  Shape das máscaras: {masks.shape}")
                    print(f"  Dtype imagens: {images.dtype}, máscaras: {masks.dtype}")
                    
                    total_patches += images.shape[0]
                except Exception as e:
                    print(f"  Erro ao ler fold {fold}: {e}")
        
        print(f"Total de patches nos folds: {total_patches}")

    @staticmethod
    def show_processed_nuinseg_info(data_dir: Path) -> None:
        """Exibir informações do NuInSeg processado (patches)."""
        dataset_path = data_dir / 'processed' / 'NuInSeg'
        
        if not dataset_path.exists():
            print(f"⚠ Dados processados NuInSeg não encontrados: {dataset_path}")
            return
        
        print("\n" + "="*50)
        print("Informações do NuInSeg Processado")
        print("="*50)
        
        images_dir = dataset_path / 'Images'
        masks_dir = dataset_path / 'Masks'
        labels_file = dataset_path / 'labels.csv'
        
        image_count = len(list(images_dir.glob('*.npy'))) if images_dir.exists() else 0
        mask_count = len(list(masks_dir.glob('*.npy'))) if masks_dir.exists() else 0
        
        print(f"Patches de imagem (.npy): {image_count}")
        print(f"Patches de máscara (.npy): {mask_count}")
        
        if labels_file.exists():
            labels_df = pd.read_csv(labels_file)
            print(f"Amostras com rótulos: {len(labels_df)}")

print('✓ Classe DatasetInfo definida com sucesso')

✓ Classe DatasetInfo definida com sucesso


In [32]:
class ProcessedSampleGenerator:
    """Gera amostras reduzidas dos datasets já processados (patchificados)."""

    def __init__(self, working_dir: Path = None, output_dir: Path = None):
        """
        Inicializar o gerador de amostras processadas.
        
        Args:
            working_dir: Diretório de trabalho (padrão: diretório atual)
            output_dir: Diretório de saída (padrão: data/samples/processed)
        """
        self.working_dir = working_dir or Path(os.getcwd())
        self.processed_dir = self.working_dir.parent / 'data' / 'processed'
        self.output_dir = output_dir or (self.working_dir.parent / 'data' / 'samples' / 'processed')
        
        logger.info(f'Processed dir: {self.processed_dir}')
        logger.info(f'Output dir: {self.output_dir}')
        
    def create_sample_dir(self, dataset_name: str) -> Path:
        """Criar estrutura de diretórios para um dataset."""
        sample_path = self.output_dir / dataset_name
        (sample_path / 'Images').mkdir(parents=True, exist_ok=True)
        (sample_path / 'Masks').mkdir(parents=True, exist_ok=True)
        logger.info(f"Diretório criado: {sample_path}")
        return sample_path
    
    def generate_monuseg_processed_samples(self, num_samples: int = 5) -> bool:
        """Gerar amostras do MoNuSeg processado (patches .npy)."""
        logger.info(f"\nGerando amostras MoNuSeg processadas (n={num_samples})...")
        
        source_dir = self.processed_dir / 'MoNuSeg'
        if not source_dir.exists():
            logger.warning(f"Diretório não encontrado: {source_dir}")
            return False
            
        try:
            labels_file = source_dir / 'labels.csv'
            if not labels_file.exists():
                logger.warning(f"Arquivo de rótulos não encontrado: {labels_file}")
                return False
                
            labels_df = pd.read_csv(labels_file)
            
            # Selecionar amostras aleatórias
            if len(labels_df) < num_samples:
                logger.warning(f"Solicitados {num_samples} mas apenas {len(labels_df)} disponíveis")
                selected_samples = labels_df['FileName'].tolist()
            else:
                selected_samples = labels_df['FileName'].sample(n=num_samples, random_state=42).tolist()
            
            sample_path = self.create_sample_dir('MoNuSeg')
            
            source_images = source_dir / 'Images'
            source_masks = source_dir / 'Masks'
            
            for filename in selected_samples:
                img_file = source_images / f'{filename}.npy'
                mask_file = source_masks / f'{filename}.npy'
                
                if img_file.exists() and mask_file.exists():
                    shutil.copy(img_file, sample_path / 'Images' / f'{filename}.npy')
                    shutil.copy(mask_file, sample_path / 'Masks' / f'{filename}.npy')
            
            sample_labels = labels_df[labels_df['FileName'].isin(selected_samples)]
            sample_labels.to_csv(sample_path / 'labels.csv', index=False)
            
            logger.info(f"✓ MoNuSeg processado: {len(selected_samples)} amostras salvas em {sample_path}")
            return True
            
        except Exception as e:
            logger.error(f"Erro ao gerar amostras MoNuSeg processadas: {e}")
            return False
    
    def generate_pannuke_processed_samples(self, num_samples: int = 10) -> bool:
        """Gerar amostras do PanNuke processado (patches de arrays .npy)."""
        logger.info(f"\nGerando amostras PanNuke processadas (n={num_samples} por fold)...")
        
        source_dir = self.processed_dir / 'PanNuke'
        if not source_dir.exists():
            logger.warning(f"Diretório não encontrado: {source_dir}")
            return False
            
        try:
            labels_file = source_dir / 'labels.csv'
            if not labels_file.exists():
                logger.warning(f"Arquivo de rótulos não encontrado: {labels_file}")
                return False
                
            labels_df = pd.read_csv(labels_file)
            
            if len(labels_df) < num_samples:
                logger.warning(f"Solicitados {num_samples} mas apenas {len(labels_df)} disponíveis")
                selected_indices = list(range(len(labels_df)))
            else:
                selected_indices = np.random.RandomState(42).choice(
                    len(labels_df), size=num_samples, replace=False
                ).tolist()
            
            sample_path = self.create_sample_dir('PanNuke')
            
            source_images = source_dir / 'Images'
            source_masks = source_dir / 'Masks'
            
            selected_rows = []
            
            for idx in selected_indices:
                row = labels_df.iloc[idx]
                filename = row['FileName']
                
                img_file = source_images / f'{filename}.npy'
                mask_file = source_masks / f'{filename}.npy'
                
                if img_file.exists() and mask_file.exists():
                    shutil.copy(img_file, sample_path / 'Images' / f'{filename}.npy')
                    shutil.copy(mask_file, sample_path / 'Masks' / f'{filename}.npy')
                    selected_rows.append(row)
            
            if selected_rows:
                sample_labels = pd.DataFrame(selected_rows)
                sample_labels = sample_labels.reset_index(drop=True)
                sample_labels.to_csv(sample_path / 'labels.csv', index=False)
            
            logger.info(f"✓ PanNuke processado: {len(selected_indices)} amostras salvas em {sample_path}")
            return True
            
        except Exception as e:
            logger.error(f"Erro ao gerar amostras PanNuke processadas: {e}")
            return False
    
    def generate_nuinseg_processed_samples(self, num_samples: int = 5) -> bool:
        """Gerar amostras do NuInSeg processado (patches .npy)."""
        logger.info(f"\nGerando amostras NuInSeg processadas (n={num_samples})...")
        
        source_dir = self.processed_dir / 'NuInSeg'
        if not source_dir.exists():
            logger.warning(f"Diretório não encontrado: {source_dir}")
            return False
            
        try:
            labels_file = source_dir / 'labels.csv'
            if not labels_file.exists():
                logger.warning(f"Arquivo de rótulos não encontrado: {labels_file}")
                return False
                
            labels_df = pd.read_csv(labels_file)
            
            if len(labels_df) < num_samples:
                logger.warning(f"Solicitados {num_samples} mas apenas {len(labels_df)} disponíveis")
                selected_indices = list(range(len(labels_df)))
            else:
                selected_indices = np.random.RandomState(42).choice(
                    len(labels_df), size=num_samples, replace=False
                ).tolist()
            
            sample_path = self.create_sample_dir('NuInSeg')
            
            source_images = source_dir / 'Images'
            source_masks = source_dir / 'Masks'
            
            selected_rows = []
            
            for idx in selected_indices:
                row = labels_df.iloc[idx]
                filename = row['FileName']
                
                img_file = source_images / f'{filename}.npy'
                mask_file = source_masks / f'{filename}.npy'
                
                if img_file.exists() and mask_file.exists():
                    shutil.copy(img_file, sample_path / 'Images' / f'{filename}.npy')
                    shutil.copy(mask_file, sample_path / 'Masks' / f'{filename}.npy')
                    selected_rows.append(row)
            
            if selected_rows:
                sample_labels = pd.DataFrame(selected_rows)
                sample_labels = sample_labels.reset_index(drop=True)
                sample_labels.to_csv(sample_path / 'labels.csv', index=False)
            
            logger.info(f"✓ NuInSeg processado: {len(selected_rows)} amostras salvas em {sample_path}")
            return True
            
        except Exception as e:
            logger.error(f"Erro ao gerar amostras NuInSeg processadas: {e}")
            return False
    
    def generate_all_processed_samples(self, monuseg: int = 5, pannuke: int = 10, nuinseg: int = 5) -> Dict[str, bool]:
        """Gerar amostras processadas para todos os datasets."""
        logger.info("\n" + "="*60)
        logger.info("Gerando Amostras Processadas")
        logger.info("="*60)
        
        results = {
            'MoNuSeg': self.generate_monuseg_processed_samples(monuseg),
            'PanNuke': self.generate_pannuke_processed_samples(pannuke),
            'NuInSeg': self.generate_nuinseg_processed_samples(nuinseg),
        }
        
        logger.info("\n" + "="*60)
        logger.info("Resumo: Amostras Processadas")
        logger.info("="*60)
        for dataset, success in results.items():
            status = "✓ SUCESSO" if success else "✗ FALHA"
            logger.info(f"{dataset}: {status}")
        
        logger.info(f"\nAmostras salvas em: {self.output_dir}")
        
        return results

print('✓ Classe ProcessedSampleGenerator definida com sucesso')

✓ Classe ProcessedSampleGenerator definida com sucesso


## 3. Configuração de Parâmetros

In [33]:
# Número de amostras para cada dataset
CONFIG = {
    'monuseg_samples': 5,      # Número de imagens MoNuSeg (padrão: 5)
    'pannuke_samples': 15,     # Número de imagens PanNuke por fold (padrão: 15)
    'nuinseg_samples': 10,      # Número de imagens NuInSeg (padrão: 10)
}

# Diretório de saída (deixe como None para usar data/samples)
OUTPUT_DIR = None

print("\n" + "="*60)
print("CONFIGURAÇÃO ATUAL")
print("="*60)
for key, value in CONFIG.items():
    print(f"{key}: {value}")
if OUTPUT_DIR:
    print(f"Output directory: {OUTPUT_DIR}")
else:
    print(f"Output directory: data/samples (padrão)")
print("="*60)


CONFIGURAÇÃO ATUAL
monuseg_samples: 5
pannuke_samples: 15
nuinseg_samples: 10
Output directory: data/samples (padrão)


## 4. Gerar Amostras Brutas

In [34]:
# Criar gerador de amostras
generator = SampleGenerator(
    working_dir=WORKING_DIRECTORY,
    output_dir=OUTPUT_DIR
)

# Gerar amostras com base na configuração
results = generator.generate_all_raw_samples(
    monuseg=CONFIG['monuseg_samples'],
    pannuke=CONFIG['pannuke_samples'],
    nuinseg=CONFIG['nuinseg_samples']
)

# Exibir resultados
print("\n" + "="*60)
print("RESULTADOS DA GERAÇÃO")
print("="*60)
for dataset, success in results.items():
    emoji = '✓' if success else '✗'
    status = 'SUCESSO' if success else 'FALHA'
    print(f"{emoji} {dataset}: {status}")

2026-05-19 18:40:42,303 - INFO - Interim dir: c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\interim
2026-05-19 18:40:42,304 - INFO - Output dir: c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\samples\raw
2026-05-19 18:40:42,306 - INFO - 
2026-05-19 18:40:42,307 - INFO - Iniciando Geração de Amostras
2026-05-19 18:40:42,309 - INFO - ============================================================
2026-05-19 18:40:42,310 - INFO - 
Gerando amostras MoNuSeg (n=5)...
2026-05-19 18:40:42,321 - INFO - Diretório criado: c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\samples\raw\MoNuSeg
2026-05-19 18:40:42,470 - INFO - ✓ MoNuSeg: 5 amostras salvas em c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\samples\raw\MoNuSeg
2026-05-19 18:40:42,471 - INFO - 
Gerando amostras PanNuke (n=15 por fold)...
2026-05-19 18:40:42,478 - INFO - Diretório criado: c:\ProjetoIma


RESULTADOS DA GERAÇÃO
✓ MoNuSeg: SUCESSO
✓ PanNuke: SUCESSO
✓ NuInSeg: SUCESSO


## 5. Gerar Amostras Processadas

In [35]:
generator_processed = ProcessedSampleGenerator(
    working_dir=WORKING_DIRECTORY,
    output_dir=None  # Usa padrão: data/samples/processed
)

results_processed = generator_processed.generate_all_processed_samples(
    monuseg=CONFIG['monuseg_samples'],
    pannuke=CONFIG['pannuke_samples'],
    nuinseg=CONFIG['nuinseg_samples']
)

# Exibir resultados
print("\n" + "="*60)
print("RESULTADOS: AMOSTRAS PROCESSADAS")
print("="*60)
for dataset, success in results_processed.items():
    emoji = '✓' if success else '✗'
    status = 'SUCESSO' if success else 'FALHA'
    print(f"{emoji} {dataset}: {status}")

print("\n" + "="*60)
print(f"Local dos dados: {generator_processed.output_dir}")
print("="*60)

2026-05-19 18:42:42,124 - INFO - Processed dir: c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\processed
2026-05-19 18:42:42,126 - INFO - Output dir: c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\samples\processed
2026-05-19 18:42:42,127 - INFO - 
2026-05-19 18:42:42,128 - INFO - Gerando Amostras Processadas
2026-05-19 18:42:42,129 - INFO - ============================================================
2026-05-19 18:42:42,129 - INFO - 
Gerando amostras MoNuSeg processadas (n=5)...
2026-05-19 18:42:42,148 - INFO - Diretório criado: c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\samples\processed\MoNuSeg
2026-05-19 18:42:42,199 - INFO - ✓ MoNuSeg processado: 5 amostras salvas em c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\samples\processed\MoNuSeg
2026-05-19 18:42:42,201 - INFO - 
Gerando amostras PanNuke processadas (n=15 por fold)...
2026-05-


RESULTADOS: AMOSTRAS PROCESSADAS
✓ MoNuSeg: SUCESSO
✓ PanNuke: SUCESSO
✓ NuInSeg: SUCESSO

Local dos dados: c:\ProjetoImagemSSD\fork_entrega\IA901-2026S1\projetos\segmentacao_de_nucleos\data\samples\processed
